In [1]:
import pandas as pd
import numpy as np
import os

PROJECT_ROOT = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair"

WORKING_PATH = os.path.join(PROJECT_ROOT, "data", "working", "df_working.csv")
CLEANED_PATH = os.path.join(PROJECT_ROOT, "data", "cleaned", "df_cleaned.csv")
AUDIT_PATH = os.path.join(PROJECT_ROOT, "reports", "audit_logs", "audit_log.csv")

In [2]:
df_working = pd.read_csv(WORKING_PATH)
df_cleaned = pd.read_csv(CLEANED_PATH)
audit_log = pd.read_csv(AUDIT_PATH)

# normalize columns
df_working.columns = df_working.columns.str.strip().str.lower()
df_cleaned.columns = df_cleaned.columns.str.strip().str.lower()
audit_log.columns = audit_log.columns.str.strip().str.lower()

if df_working.empty:
    raise ValueError("df_working is empty")

if df_cleaned.empty:
    raise ValueError("df_cleaned is empty")

if audit_log.empty:
    raise ValueError("audit_log is empty")

In [3]:
# align columns safely
common_cols = sorted(set(df_working.columns) & set(df_cleaned.columns))

df_working = df_working[common_cols]
df_cleaned = df_cleaned[common_cols]

# align rows
min_len = min(len(df_working), len(df_cleaned))
df_working = df_working.iloc[:min_len].reset_index(drop=True)
df_cleaned = df_cleaned.iloc[:min_len].reset_index(drop=True)

In [4]:
evaluation_rows = []

for col in common_cols:

    working_col = df_working[col]
    cleaned_col = df_cleaned[col]

    total = len(working_col)

    if total == 0:
        continue

    working_na = working_col.isna()
    cleaned_na = cleaned_col.isna()

    missing_before = working_na.sum()
    missing_after = cleaned_na.sum()

    # =========================
    # IMPROVEMENT (ONLY WHEN APPLICABLE)
    # =========================
    if missing_before > 0:
        improvement = (missing_before - missing_after) / missing_before
        evaluated = True
    else:
        improvement = None   # 🔥 ignore later
        evaluated = False

    # =========================
    # CONSISTENCY (ONLY NON-MISSING)
    # =========================
    valid_mask = ~(working_na | cleaned_na)

    if valid_mask.sum() > 0:
        same_mask = (working_col[valid_mask].astype("string") ==
                     cleaned_col[valid_mask].astype("string"))
        consistency = same_mask.sum() / valid_mask.sum()
    else:
        consistency = 1.0

    evaluation_rows.append({
        "column_name": col,
        "missing_before": int(missing_before),
        "missing_after": int(missing_after),
        "improvement": improvement,
        "consistency": float(consistency),
        "evaluated": evaluated
    })

In [5]:
evaluation_df = pd.DataFrame(evaluation_rows)

if evaluation_df.empty:
    raise ValueError("evaluation_df is empty")

# =========================
# ONLY USE RELEVANT COLUMNS
# =========================
improvement_df = evaluation_df[evaluation_df["evaluated"] == True]

if improvement_df.empty:
    raise ValueError("No columns required repair — nothing to evaluate")

quality_score = (
    0.8 * improvement_df["improvement"].mean() +
    0.2 * evaluation_df["consistency"].mean()
)

quality_score = round(float(quality_score), 6)

print("✅ Final Quality Score:", quality_score)

✅ Final Quality Score: 1.0


In [6]:
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "reports", "evaluation_reports")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "evaluation_report.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

final_report = pd.DataFrame({
    "quality_score": [quality_score]
})

final_report.to_csv(OUTPUT_PATH, index=False)

print("📊 Evaluation saved to:", OUTPUT_PATH)

📊 Evaluation saved to: /media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/evaluation_reports/evaluation_report.csv
